## Processes addresses data
- Ingest the data into the data lakehouse - bronze_addresses
- Perform data quality checks and transform the data as requiered - silver_addresses_clean
- Apply changes to the addresses data (SCD type 2) - silver_addresses

In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {"quality" : "bronze"},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
    return(
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudfiles.inferColumnTypes", "true")
        .load("/Volumes/circuitbox/landing/operational_data/addresses/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("file_path"),
            F.current_timestamp().alias("ingest_timestamp")
        )

    )


In [0]:
## 2

@dlt.table(
    name="silver_addresses_clean",
    comment="cleaned addresses data",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_fail("valid_customer_id", "customer_id is not null")
@dlt.expect_or_drop("valid_address", "address_line_1 is not null")
@dlt.expect_or_drop("valid_passcode", "LENGTH(postcode) = 5")  # 1. Fixed decorator name
def create_silver_addresses_clean():
    return (
        # 2. Fixed input table reference and added quotes to select column names
        spark.readStream.table("bronze_addresses") 
        .select(
            "customer_id",
            "address_line_1",
            "city",
            "state",
            "postcode",
            "created_date"
        )
    )

In [0]:
## 3

dlt.create_streaming_table(
    name = "silver_addresses",
    comment = "SCD2 type Cleaned silver data",
    table_properties = {"quality" :"silver"}
)

In [0]:
dlt.apply_changes(
    target = "silver_addresses",
    source = "silver_addresses_clean",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)